# Single-Name Cross-Sectional Momentum (Real Stocks)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/lab/blob/main/notebooks/single_name_momentum.ipynb)

The in-browser playground forms momentum from *industry* portfolios. Here we do the real thing:
download **individual stock** prices, rank the cross-section each month on trailing 12-1 returns,
and form a long-short book — Jegadeesh & Titman (1993) at the single-name level — then test it out of
sample. Uses a liquid large-cap universe so it runs quickly in Colab.

In [ ]:
!pip -q install yfinance

In [ ]:
import yfinance as yf, pandas as pd, numpy as np, matplotlib.pyplot as plt

# A liquid large-cap universe (swap in your own tickers).
TICKERS = ("AAPL MSFT AMZN GOOGL META NVDA JPM JNJ V PG XOM HD BAC KO PFE CSCO INTC "
           "WMT DIS MCD CVX ABT NKE ORCL CRM TXN QCOM COST AMD CAT GE MMM IBM GS").split()
px = yf.download(TICKERS, start="2005-01-01", auto_adjust=True, progress=False)["Close"]
rets = px.resample("ME").last().pct_change()
print("universe:", rets.shape[1], "stocks,", rets.shape[0], "months")

## Form the long-short momentum book

In [ ]:
# Trailing 12-1 month signal (skip the most recent month to avoid short-term reversal).
sig = (1 + rets).rolling(12).apply(np.prod, raw=True) / (1 + rets) - 1

# Each month: long the top quintile, short the bottom quintile (equal weight), held next month.
def cross_sectional_book(sig, q=0.2):
    w = pd.DataFrame(0.0, index=sig.index, columns=sig.columns)
    for dt, row in sig.dropna(how="all").iterrows():
        r = row.dropna()
        if len(r) < 10: continue
        k = max(1, int(len(r) * q))
        order = r.sort_values()
        w.loc[dt, order.index[-k:]] =  1.0 / k
        w.loc[dt, order.index[:k]]  = -1.0 / k
    return w.shift(1)

w = cross_sectional_book(sig)
strat = (rets * w).sum(axis=1).dropna()
sharpe = lambda x: np.sqrt(12) * x.mean() / x.std()
print("full-sample monthly Sharpe (annualized):", round(sharpe(strat), 2))

## Out-of-sample split

In [ ]:
split = 2015   # train/test boundary (yfinance history is recent, so we split mid-sample)
yrs = strat.index.year
ins, oos = strat[yrs < split], strat[yrs >= split]
print(f"IS (<{split}) Sharpe: {sharpe(ins):+.2f}    OOS (>={split}) Sharpe: {sharpe(oos):+.2f}")

(1 + strat).cumprod().plot(logy=True, figsize=(9,4), title="Single-name cross-sectional momentum")
plt.axvline(pd.Timestamp(f"{split}-01-01"), color="crimson", ls="--", label="OOS split")
plt.legend(); plt.tight_layout(); plt.show()

## Your turn
Expand the universe, change the quintile cutoff, add a one-month skip or transaction costs, or
swap momentum for a value/quality signal. This downloads fresh data every run, so you can test
genuinely out of sample on the latest months.